# Homework 5
## Beks Spicuglia

### AI Assistance Disclosure

I used ChatGPT to generate and revise portions of the Python code, explain programming steps, and troubleshoot errors. Assistance included scraping and cleaning GDP data, assigning geographic regions, creating the Plotly stacked bar chart, adapting the course’s MRICloud example into a Sankey diagram, handling duplicate region names, and adding volume-conservation checks. ChatGPT also assisted with HTML export, GitHub Pages setup, and checking the assignment files. The MRICloud data and hierarchy sources come from the course example; the GDP source is Wikipedia.

## Part 1: GDP by Country and Region

I use the Wikipedia revision linked in the assignment, select IMF forecasts, and stack countries within the table's UN regions. Rows without numeric IMF forecasts and the World aggregate are excluded. GDP values are converted from millions to trillions of US dollars.

In [1]:
# [wAAI ChatGPT] Read the assignment's Wikipedia page and identify the IMF table.
import requests
from io import StringIO
import pandas as pd
import plotly.express as px

url = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_countries_by_GDP_(nominal)&oldid=1187446467"
)

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)
response.raise_for_status()

# Read all HTML tables on the page
page_tables = pd.read_html(StringIO(response.text))

# Identify the GDP table by its column headings
gdp_raw = None

for table in page_tables:
    headings = " ".join(str(col) for col in table.columns).lower()

    if "country" in headings and (
        "imf" in headings or "international monetary fund" in headings
    ):
        gdp_raw = table
        break

if gdp_raw is None:
    for number, table in enumerate(page_tables):
        print(f"Table {number}: {table.columns.tolist()}")
    raise ValueError("Could not identify the IMF table from its headings.")

display(gdp_raw.head())
print("Column names:")
print(gdp_raw.columns)

Country/Territory UN region IMF[1][13]            World Bank[14]             \
  Country/Territory UN region   Forecast       Year       Estimate       Year   
0             World         —  104476432       2023      100562011       2022   
1     United States  Americas   26949643       2023       25462700       2022   
2             China      Asia   17700899  [n 1]2023       17963171  [n 3]2022   
3           Germany    Europe    4429838       2023        4072192       2022   
4             Japan      Asia    4230862       2023        4231141       2022   

  United Nations[15]             
            Estimate       Year  
0           96698005       2021  
1           23315081       2021  
2           17734131  [n 1]2021  
3            4259935       2021  
4            4940878       2021

Column names:
MultiIndex([( 'Country/Territory', 'Country/Territory'),
            (         'UN region',         'UN region'),
            (        'IMF[1][13]',          'Forecast'),
            (        'IMF[1][13]',              'Year'),
            (    'World Bank[14]',          'Estimate'),
            (    'World Bank[14]',              'Year'),
            ('United Nations[15]',          'Estimate'),
            ('United Nations[15]',              'Year')],
           )


In [2]:
# [wAAI ChatGPT] Select and clean IMF forecasts and UN regions.

# The table has two-row column headings.
imf_column = next(
    col for col in gdp_raw.columns
    if col[0].startswith("IMF") and col[1] == "Forecast"
)

gdp = gdp_raw[
    [
        ("Country/Territory", "Country/Territory"),
        ("UN region", "UN region"),
        imf_column
    ]
].copy()

gdp.columns = ["country", "region", "gdp_millions"]

# Remove footnote markers and surrounding spaces.
for column in ["country", "region"]:
    gdp[column] = (
        gdp[column]
        .astype("string")
        .str.replace(r"\[.*?\]", "", regex=True)
        .str.strip()
    )

# Convert IMF forecasts to numbers.
gdp["gdp_millions"] = pd.to_numeric(
    gdp["gdp_millions"]
    .astype(str)
    .str.replace(r"\[.*?\]", "", regex=True)
    .str.replace(",", "", regex=False)
    .str.strip(),
    errors="coerce"
)

# Exclude the World total and rows without IMF forecasts.
gdp = gdp.loc[gdp["country"] != "World"].dropna(
    subset=["gdp_millions"]
).copy()

# Convert millions to trillions of US dollars.
gdp["gdp_trillions"] = gdp["gdp_millions"] / 1_000_000

display(gdp.head())
print(f"Countries/territories with IMF forecasts: {len(gdp)}")

display(
    gdp.groupby("region", as_index=False)
    .agg(
        countries=("country", "count"),
        gdp_trillions=("gdp_trillions", "sum")
    )
    .sort_values("gdp_trillions", ascending=False)
)

,country,region,gdp_millions,gdp_trillions
1,United States,Americas,26949643.0,26.949643
2,China,Asia,17700899.0,17.700899
3,Germany,Europe,4429838.0,4.429838
4,Japan,Asia,4230862.0,4.230862
5,India,Asia,3732224.0,3.732224


Countries/territories with IMF forecasts: 191


,region,countries,gdp_trillions
2,Asia,46,38.446685
1,Americas,36,35.702210
3,Europe,42,25.418934
0,Africa,53,2.867253
4,Oceania,14,1.980134


In [3]:
# [wAAI ChatGPT] Plot countries within UN regions and export interactive HTML.
# Order regions by total GDP
region_order = (
    gdp.groupby("region")["gdp_trillions"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

# Each country is a segment within its region's bar
fig_gdp = px.bar(
    gdp.sort_values("gdp_trillions", ascending=False),
    x="region",
    y="gdp_trillions",
    color="country",
    category_orders={"region": region_order},
    title="GDP by Country and UN Region — IMF Forecasts",
    labels={
        "region": "UN region",
        "gdp_trillions": "GDP (trillions of US dollars)",
        "country": "Country/Territory"
    },
    hover_name="country",
    height=650
)

fig_gdp.update_layout(
    barmode="stack",
    template="plotly_white",
    showlegend=False
)

fig_gdp.update_traces(
    hovertemplate=(
        "<b>%{hovertext}</b><br>"
        "Region: %{x}<br>"
        "GDP: US$%{y:.3f} trillion"
        "<extra></extra>"
    )
)

fig_gdp.show()

# Save a standalone interactive webpage
fig_gdp.write_html("stacked_bar.html", include_plotlyjs=True)

print("Saved stacked_bar.html")

Saved stacked_bar.html


## Part 2: MRICloud Sankey Diagram
Subject 127, Type 1: brain-region volumes from intracranial volume through five anatomical levels.

In [4]:
# [wAAI ChatGPT] Prepare the course MRICloud data and resolve duplicate region names.
import pandas as pd

# Load the anatomical hierarchy
lookup_url = (
    "https://raw.githubusercontent.com/bcaffo/"
    "MRIcloudT1volumetrics/master/inst/extdata/"
    "multilevel_lookup_table.txt"
)

lookup = pd.read_csv(lookup_url, sep="\t")

lookup = lookup.rename(columns={
    "modify": "roi",
    "modify.1": "level4",
    "modify.2": "level3",
    "modify.3": "level2",
    "modify.4": "level1"
})

lookup = lookup[
    ["roi", "level4", "level3", "level2", "level1"]
].drop_duplicates()

# Check that each region has a single hierarchy path
assert not lookup["roi"].duplicated().any(), \
    "A region has conflicting hierarchy paths."

# Load the brain-volume data
data_url = (
    "https://raw.githubusercontent.com/smart-stats/"
    "ds4bio_book/main/book/assetts/kirby21AllLevels.csv"
)

brain = pd.read_csv(data_url)

# Select subject 127, Type 1, finest anatomical level
subject = brain.loc[
    (brain["id"] == 127)
    & (brain["type"] == 1)
    & (brain["level"] == 5),
    ["roi", "volume"]
].copy()

# Combine volumes for repeated region names
subject = subject.groupby(
    "roi", as_index=False
)["volume"].sum()

volume_before_merge = subject["volume"].sum()

# Attach the anatomical hierarchy
subject = subject.merge(
    lookup,
    on="roi",
    how="left",
    validate="one_to_one"
)

# Add the intracranial volume root
subject["icv"] = "ICV"

hierarchy = [
    "icv", "level1", "level2", "level3", "level4", "roi"
]

# Check for missing hierarchy information or changed totals
assert not subject[hierarchy].isna().any().any(), \
    "Some regions are missing hierarchy information."

assert subject["volume"].sum() == volume_before_merge, \
    "The merge changed the total volume."

display(subject.head())
print("Number of finest-level region names:", len(subject))
print("Total volume:", subject["volume"].sum())

,roi,volume,level4,level3,level2,level1,icv
0,ACR_L,12600,ant_DPWM_L,AnteriorWM_L,WhiteMatter_L,Telencephalon_L,ICV
1,ACR_R,13280,ant_DPWM_R,AnteriorWM_R,WhiteMatter_R,Telencephalon_R,ICV
2,AGWM_L,3636,PeripheralParietalWM_L,PosteriorWM_L,WhiteMatter_L,Telencephalon_L,ICV
3,AGWM_R,5099,PeripheralParietalWM_R,PosteriorWM_R,WhiteMatter_R,Telencephalon_R,ICV
4,AG_L,7430,AG_L,Parietal_L,CerebralCortex_L,Telencephalon_L,ICV


Number of finest-level region names: 274
Total volume: 1378405


In [5]:
# [wAAI ChatGPT] Build the Sankey hierarchy and check volume conservation.
import plotly.graph_objects as go

# Give each node a unique ID based on its full hierarchy path.
# This keeps repeated names at different levels separate.
node_ids = {}
node_labels = []
node_levels = []

for depth in range(len(hierarchy)):
    for path in subject[hierarchy[:depth + 1]].itertuples(
        index=False, name=None
    ):
        if path not in node_ids:
            node_ids[path] = len(node_labels)
            node_labels.append(path[-1])
            node_levels.append(depth)

# Sum volumes for each connection between adjacent levels
sources = []
targets = []
values = []

for depth in range(len(hierarchy) - 1):
    path_columns = hierarchy[:depth + 2]

    connections = (
        subject.groupby(path_columns, as_index=False)["volume"]
        .sum()
    )

    for row in connections.itertuples(index=False, name=None):
        child_path = tuple(row[:-1])
        parent_path = child_path[:-1]

        sources.append(node_ids[parent_path])
        targets.append(node_ids[child_path])
        values.append(float(row[-1]))

# Check that each hierarchy transition carries the total volume
total_volume = float(subject["volume"].sum())

for depth in range(len(hierarchy) - 1):
    level_total = sum(
        value
        for source, value in zip(sources, values)
        if node_levels[source] == depth
    )
    assert abs(level_total - total_volume) < 0.01

# Use a different color for each hierarchy level
level_colors = [
    "#264653", "#2a9d8f", "#457b9d",
    "#8ab17d", "#e9c46a", "#f4a261"
]

fig_sankey = go.Figure(
    go.Sankey(
        arrangement="snap",
        node=dict(
            label=node_labels,
            color=[level_colors[d] for d in node_levels],
            pad=12,
            thickness=15,
            line=dict(color="white", width=0.5)
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            hovertemplate=(
                "%{source.label} → %{target.label}<br>"
                "Volume: %{value:,.0f} mm³"
                "<extra></extra>"
            )
        )
    )
)

fig_sankey.update_layout(
    title_text=(
        "MRICloud Brain-Volume Hierarchy — Subject 127, Type 1"
        "<br><sup>ICV → Level 1 → Level 2 → Level 3 "
        "→ Level 4 → Level 5 (ROI)</sup>"
    ),
    font_size=10,
    width=1600,
    height=2400,
    margin=dict(l=20, r=20, t=100, b=20)
)

fig_sankey.show()

# Save the interactive diagram as a standalone webpage
fig_sankey.write_html("sankey.html", include_plotlyjs=True)

print("Saved sankey.html")
print("Hierarchy transitions checked: total volume preserved.")

Saved sankey.html
Hierarchy transitions checked: total volume preserved.


## Part 3: Publicly Hosted Sankey Diagram

[View the interactive MRICloud Sankey diagram](https://rspicuglia.github.io/hw5-sankey/sankey.html)